In [19]:
from transformers import AutoModelForTokenClassification , AutoTokenizer
import torch
from transformers import AutoConfig

In [13]:
model_name = "dslim/bert-base-NER"

In [14]:
model = AutoModelForTokenClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [25]:
config = AutoConfig.from_pretrained(model_name)
print("parameters " , config.pad_token_id)
print(config.add_cross_attention)
print(config.architectures)
print(config.id2label)
print(config.transformers_version)

parameters  0
False
['BertForTokenClassification']
{0: 'O', 1: 'B-MISC', 2: 'I-MISC', 3: 'B-PER', 4: 'I-PER', 5: 'B-ORG', 6: 'I-ORG', 7: 'B-LOC', 8: 'I-LOC'}
None


In [41]:
class Ner:
  def __init__(self):
    self.model = AutoModelForTokenClassification.from_pretrained(model_name)
    self.tokenizer = AutoTokenizer.from_pretrained(model_name)
    self.id2label = self.model.config.id2label

  def tokenize(self,text):
    return self.tokenizer(text , padding =True , truncation = True , return_tensors = 'pt')

  def predict(self , text):
    inputs = self.tokenize(text)
    outputs = self.model(**inputs)
    logits = outputs.logits
    probs = torch.nn.functional.softmax(logits , dim =-1 )
    preds_indices = torch.argmax(probs , dim =-1)

    predicted_labels = []
    predicted_confidences = []
    for i in range(preds_indices.shape[1]):
        token_pred_idx = preds_indices[0, i].item()
        label = self.id2label[token_pred_idx]
        confidence = probs[0, i, token_pred_idx].item()
        predicted_labels.append(label)
        predicted_confidences.append(confidence)
    tokens = self.tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    return tokens, predicted_labels, predicted_confidences

  def result(self,text):
    tokens, labels, confidences = self.predict(text)
    print(f"Text : {text}")
    print("Detected Entities:")
    for token, label, conf in zip(tokens, labels, confidences):
        if token.startswith("##"):
            token_display = token[2:]
        else:
            token_display = token
        if label != 'O' and token not in ['[CLS]', '[SEP]']:
            print(f"  Token: '{token_display}', Label: '{label}', Confidence: {conf:.4f}")

In [42]:
mod= Ner()
text = "farah is going to visit many place like the Lahore , Multan"
mod.result(text)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Text : farah is going to visit many place like the Lahore , Multan
Detected Entities:
  Token: 'far', Label: 'B-PER', Confidence: 0.9886
  Token: 'Lahore', Label: 'B-LOC', Confidence: 0.9992
  Token: 'Mu', Label: 'B-LOC', Confidence: 0.9981
  Token: 'lta', Label: 'B-LOC', Confidence: 0.9918
  Token: 'n', Label: 'I-LOC', Confidence: 0.9934
